<a href="https://colab.research.google.com/github/jlam-data/assistente_voz_sql/blob/main/Boot_Bradesco_Whisper_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este é um assistente de voz criado para captar pedidos de usuário quanto a consultas de dados. Ele capta o que o usuário diz, transforma em uma consulta SQL (query) e informa o que vai fazer, em uma linguagem amigável a pessoas que não têm tanto conhecimento técnico. Assim, em vez de dizer "SELECT MAX(valor_total) FROM tabela_vendas", o assistente de voz vai dizer: "Selecionei o maior valor da coluna valor_total na tabela_vendas".

In [ ]:
language = 'pt'

In [ ]:
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 5 segundos)
print('Ouvindo...\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

Ouvindo...



<IPython.core.display.Javascript object>

In [ ]:
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import whisper

model = whisper.load_model("small")

result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"]
print(transcription)

 Qual é o menor valor total da tabela vendas?


In [ ]:
!pip install openai

In [ ]:
import os
from google.colab import userdata
from openai import OpenAI

# 1. Configuração do Cliente
try:
    api_key = userdata.get('boot')
    client = OpenAI(api_key=api_key)
except Exception as e:
    print(f"Certifique-se de que o Secret 'boot' está ativo: {e}")

# Pegamos a transcrição do Whisper (vinda do bloco anterior do seu Colab)
transcription = result["text"]

# 2. Definição da Instrução do Sistema (O nosso "Mapa" de Dados)
instrucao_sistema = """
Você é um especialista em SQL. Traduza os pedidos do usuário para comandos SQL.
Considere o seguinte schema do banco de dados:

1. tabela_cliente: id_cliente (PK), nome_cliente
2. tabela_venda: id_venda (PK), data_venda, id_cliente (FK), valor_total, id_forma_pagamento (FK)
3. tabela_catalogo_produtos: id_produto (PK), nome_produto, custo_fornecedor
4. tabela_vendas_produtos: id_venda_produto (PK), id_venda (FK), id_produto (FK), id_cliente (FK), desconto
5. tabela_financeiro: id_forma_pagamento (PK), forma_pagamento

Sua resposta deve seguir estritamente este formato:
SQL: SELECT MAX(valor_total) FROM tabela_venda
EXPLICAÇÃO: Fiz uma consulta e identifiquei que o maior valor_total d tabela_venda é de cinco mil reais.
"""

# 3. Chamada da API com a estrutura de mensagens (System + User)
response = client.chat.completions.create(
    model="gpt-4", # Ou "gpt-3.5-turbo", mas o gpt-4 é melhor para JOINS complexos
    messages=[
        {"role": "system", "content": instrucao_sistema},
        {"role": "user", "content": transcription}
    ]
)

# 4. Extração da resposta e Tratamento de Texto
chatgpt_response = response.choices[0].message.content
print(f"Resposta Completa da IA:\n{chatgpt_response}")

# Usamos o split para separar o que vai ser exibido do que vai ser falado
if "EXPLICAÇÃO:" in chatgpt_response:
    partes = chatgpt_response.split("EXPLICAÇÃO:")
    codigo_sql = partes[0].replace("SQL:", "").strip()
    texto_para_audio = partes[1].strip()
else:
    texto_para_audio = chatgpt_response

print(f"\nO que será lido pelo gTTS: {texto_para_audio}")

Resposta Completa da IA:
SQL: SELECT MIN(valor_total) FROM tabela_venda
EXPLICAÇÃO: Esta consulta retornará o valor mínimo da coluna 'valor_total' da tabela 'tabela_venda'.

O que será lido pelo gTTS: Esta consulta retornará o valor mínimo da coluna 'valor_total' da tabela 'tabela_venda'.


In [ ]:
!pip install gtts
from gtts import gTTS

gtts_objetc = gTTS(text=texto_para_audio, lang=language, slow=False)

response_audio = "/content/response_audio.wav"
gtts_objetc.save(response_audio)

display(Audio(response_audio, autoplay=True))

Até aqui foi feito o seguinte: por voz, o usuário faz um pedido, uma consulta por voz relacionada a uma tabela (momento speech to text). O assistente capta e transforma esse pedido em um comando SQL e informa por voz o que será feito (momento text to speech). Para desenvolvimento futuro, seria interessante trabalhar na interoperabilidade deste código com banco de dados, de modo a realizar consultas SQL e retornar com as respostas da solicitação.

Assim, a ideia é deixar o trabalho pronto por módulos. O assistente de voz já está pronto e testado. Ele consegue captar o pedido, transformar em uma query SQL e retornar o feedback numa linguagem amigável (não tão técnica) para o usuário, com voz (text to speech).

A próxima etapa tem uma prévia abaixo, que é a criação de tabelas "dummies" para testes com o banco de dados.

In [ ]:
## CRIANDO TABELAS DE SIMULAÇÃO (para momento 2)
# Essas tabelas servirão para testar o código que geramos, já que o propósito é criar consultas SQL com base em comandos de voz

!pip install faker
import pandas as pd
import random
from faker import Faker

# Inicializamos o gerador de dados falsos para nomes BR e US
fake_br = Faker('pt_BR')
fake_us = Faker('en_US')

# 1. TABELA_FINANCEIRO (Formas de Pagamento)
financeiro_data = [
    {'id_forma_pagamento': 'F01', 'forma_pagamento': 'Cartão de Crédito'},
    {'id_forma_pagamento': 'F02', 'forma_pagamento': 'Boleto Bancário'},
    {'id_forma_pagamento': 'F03', 'forma_pagamento': 'Pix'},
    {'id_forma_pagamento': 'F04', 'forma_pagamento': 'Dinheiro'}
]
df_financeiro = pd.DataFrame(financeiro_data);

# 2. TABELA_CLIENTE (80 clientes mistos)
clientes = []
for i in range(1, 81):
    nacionalidade = random.choice(['BR', 'US'])
    nome = fake_br.name() if nacionalidade == 'BR' else fake_us.name()
    clientes.append({'id_cliente': f'C{i:03d}', 'nome_cliente': nome})
df_cliente = pd.DataFrame(clientes)

# 3. TABELA_CATALOGO_PRODUTOS (15 produtos simulando tecnologia/escritório)
produtos_nomes = ['Laptop Pro', 'Mouse Sem Fio', 'Monitor 4K', 'Teclado Mecânico', 'Cadeira Ergonômica',
                  'Webcam HD', 'Headset Gamer', 'HD Externo 1TB', 'Suporte Notebook', 'Cabo HDMI']
catalogo = []
for i, nome in enumerate(produtos_nomes):
    catalogo.append({
        'id_produto': f'P{i+1:02d}',
        'nome_produto': nome,
        'custo_fornecedor': round(random.uniform(50.0, 2500.0), 2)
    })
df_catalogo = pd.DataFrame(catalogo)

# 4. TABELA_VENDA (100 vendas)
vendas = []
for i in range(1, 101):
    vendas.append({
        'id_venda': f'V{i:04d}',
        'data_venda': fake_br.date_between(start_date='-1y', end_date='today'),
        'id_cliente': random.choice(df_cliente['id_cliente']),
        'valor_total': 0, # Será calculado depois
        'id_forma_pagamento': random.choice(df_financeiro['id_forma_pagamento'])
    })
df_venda = pd.DataFrame(vendas)

# 5. TABELA_VENDAS_PRODUTOS (150 itens vendidos)
vendas_produtos = []
for i in range(1, 151):
    venda_ref = random.choice(df_venda.to_dict('records'))
    prod_ref = random.choice(df_catalogo.to_dict('records'))
    vendas_produtos.append({
        'id_venda_produto': f'VP{i:04d}',
        'id_venda': venda_ref['id_venda'],
        'id_produto': prod_ref['id_produto'],
        'id_cliente': venda_ref['id_cliente'],
        'desconto': random.choice([0, 5.0, 10.0, 15.0]) # Desconto em valor
    })
df_vendas_produtos = pd.DataFrame(vendas_produtos)

# Recalcular valor_total na tabela_venda com base nos produtos e descontos
for idx, row in df_venda.iterrows():
    itens = df_vendas_produtos[df_vendas_produtos['id_venda'] == row['id_venda']]
    total = 0.0 # Initialize as float
    for _, item in itens.iterrows():
        # Ensure preco is a scalar by using .item() on the single-element numpy array
        preco = df_catalogo[df_catalogo['id_produto'] == item['id_produto']]['custo_fornecedor'].values.item() * 1.3 # Margem de lucro
        total += (preco - item['desconto'])
    df_venda.at[idx, 'valor_total'] = round(max(total, 0), 2)

# Exibindo as primeiras linhas de uma tabela para conferir
print("Tabela Cliente (Amostra):")
display(df_cliente.head())

Tabela Cliente (Amostra):


/tmp/ipykernel_16637/880869798.py:76: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4139.84' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_venda.at[idx, 'valor_total'] = round(max(total, 0), 2)


,id_cliente,nome_cliente
0,C001,Kathleen Rice
1,C002,Letícia Nascimento
2,C003,Sra. Mariana Melo
3,C004,Bernardo Santos
4,C005,Lorenzo Carvalho
